<a href="https://colab.research.google.com/github/miriam-silva/PLN/blob/main/Aula_7_Passo_a_Passo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalação e download de recursos (rodar 1x)

!pip install -q spacy scikit-learn pandas
!python -m spacy download pt_core_news_sm -q

import spacy
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

nlp = spacy.load("pt_core_news_sm")
print("Ambiente configurado com sucesso.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 84.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Ambiente configurado com sucesso.


In [2]:
corpus_leads_kdt = [
    "Quero saber o valor do condomínio do apartamento no centro.",
    "Aceita financiamento pela Caixa Econômica?",
    "Qual o valor de entrada para comprar esse imóvel?",
    "O apartamento já está pronto para morar ou ainda em construção?",
    "Vocês trabalham com financiamento bancário para a compra?",
    "Ainda está disponível para locação?",
    "Qual o valor do aluguel e do IPTU na região?",
    "Posso agendar uma visita para ver a casa alugada?",
    "O contrato de aluguel é de quanto tempo?",
    "Aceita fiador ou seguro fiança para locação?",
]

for i, msg in enumerate(corpus_leads_kdt, 1):
    print(f"Lead {i}: {msg}")

Lead 1: Quero saber o valor do condomínio do apartamento no centro.
Lead 2: Aceita financiamento pela Caixa Econômica?
Lead 3: Qual o valor de entrada para comprar esse imóvel?
Lead 4: O apartamento já está pronto para morar ou ainda em construção?
Lead 5: Vocês trabalham com financiamento bancário para a compra?
Lead 6: Ainda está disponível para locação?
Lead 7: Qual o valor do aluguel e do IPTU na região?
Lead 8: Posso agendar uma visita para ver a casa alugada?
Lead 9: O contrato de aluguel é de quanto tempo?
Lead 10: Aceita fiador ou seguro fiança para locação?


In [3]:
frase_exemplo = "O cliente João Silva entrou em contato sobre um apartamento em São Paulo financiado pela Caixa Econômica Federal."
doc_exemplo = nlp(frase_exemplo)

for ent in doc_exemplo.ents:
    print(f"{ent.text:30} -> {ent.label_}")

João Silva                     -> PER
São Paulo                      -> LOC
Caixa Econômica Federal        -> ORG


In [4]:
def lematizar(texto):
    doc = nlp(texto)
    return " ".join([token.lemma_.lower() for token in doc if not token.is_punct and not token.is_stop])

corpus_lematizado = [lematizar(doc) for doc in corpus_leads_kdt]

vectorizer_tfidf = TfidfVectorizer()
matriz_tfidf = vectorizer_tfidf.fit_transform(corpus_lematizado)
palavras = vectorizer_tfidf.get_feature_names_out()

print("Vocabulário:", list(palavras))

Vocabulário: ['aceita', 'agendar', 'alugar', 'aluguel', 'apartamento', 'bancário', 'caixa', 'casa', 'centro', 'compra', 'comprar', 'condomínio', 'construção', 'contrato', 'disponível', 'econômica', 'entrada', 'fiador', 'fiançar', 'financiamento', 'imóvel', 'iptu', 'locação', 'morar', 'pronto', 'região', 'seguro', 'trabalhar', 'visita']


In [5]:
def palavras_chave(indice_doc, top_n=3):
    linha = matriz_tfidf[indice_doc].toarray()[0]
    top_indices = linha.argsort()[::-1][:top_n]
    return [(palavras[i], round(linha[i], 2)) for i in top_indices if linha[i] > 0]

for i, msg in enumerate(corpus_leads_kdt):
    print(f"Lead {i+1}: {msg}")
    print("  Palavras-chave:", palavras_chave(i))

Lead 1: Quero saber o valor do condomínio do apartamento no centro.
  Palavras-chave: [('centro', np.float64(0.61)), ('condomínio', np.float64(0.61)), ('apartamento', np.float64(0.52))]
Lead 2: Aceita financiamento pela Caixa Econômica?
  Palavras-chave: [('econômica', np.float64(0.54)), ('caixa', np.float64(0.54)), ('aceita', np.float64(0.46))]
Lead 3: Qual o valor de entrada para comprar esse imóvel?
  Palavras-chave: [('imóvel', np.float64(0.58)), ('entrada', np.float64(0.58)), ('comprar', np.float64(0.58))]
Lead 4: O apartamento já está pronto para morar ou ainda em construção?
  Palavras-chave: [('pronto', np.float64(0.52)), ('morar', np.float64(0.52)), ('construção', np.float64(0.52))]
Lead 5: Vocês trabalham com financiamento bancário para a compra?
  Palavras-chave: [('trabalhar', np.float64(0.52)), ('bancário', np.float64(0.52)), ('compra', np.float64(0.52))]
Lead 6: Ainda está disponível para locação?
  Palavras-chave: [('disponível', np.float64(0.76)), ('locação', np.float64